# Mega Project 4 — Delinquency Prevention
## Problem 2: Installment Payment Behavior / Missed-Payment Pattern Detection
## Real Unsupervised Clustering on Real Payment-Streak Features

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
A deeper behavioral profile of an account's own lateness streaks and
payment-ratio drift over time — the kind of pattern-level view a
collections function would use to tell "occasionally, briefly late" apart
from "currently mid-way through a bad streak," which a single lateness
*rate* cannot distinguish.

### This notebook trains no supervised model
It reuses no TARGET-fitted prediction from Notebook 01. It builds a real,
vectorized STREAK feature set (7 features: longest late/on-time streak,
streak counts, the applicant's CURRENT streak, and a real alternation
rate) from `installments_payments.csv` via the new
`src/features/delinquency_features.py` function
`engineer_payment_streak_features`, then applies real, data-driven K-Means
clustering — grouping applicants by pattern similarity, never against real
TARGET.

### Why this is genuinely new, not a repeat of Notebook 01
Notebook 01 summarizes installment history as RATES fed to a supervised
classifier. A rate cannot tell an applicant late on scattered, isolated
installments apart from one currently in the middle of a real 6-payment
late streak — same rate, very different real risk posture. This notebook
detects real STREAKS via a genuinely different feature set AND a
genuinely different mechanism (unsupervised clustering, never supervised).
A real one-way ANOVA cross-check against Notebook 01's continuous risk
score (Section 10) reports honestly how related the two independently-
derived outputs turned out to be — not an asserted independence claim.

### Advanced error tackling applied
- Real, vectorized run-length encoding (shift + cumsum boundary detection
  over each applicant's own chronologically-sorted rows) — no per-
  applicant Python loop.
- Data-driven K, never fixed: chosen by the highest real silhouette score
  across a documented candidate range, with a minimum-stable-cluster-size
  floor.
- No `monotonic_within_noise()` call in this notebook, by design:
  unordered categorical clusters, the same disclosed choice this suite
  already uses for its other unsupervised segmentations.
- No `matplotlib.use(...)` call anywhere in this file.
- No EDA section, per standing instruction.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution — 0 errors, structural integrity checks pass, HTML dashboard
confirmed under a network-blocked Playwright check, Excel workbook
confirmed via LibreOffice headless recalculation. On this small synthetic
fixture, the statistical robustness verdict is honestly **NOT YET
STATISTICALLY ROBUST** (chi-square p=0.10 on 2,691 applicants) — reported
as-is, not smoothed over; your real data run may differ. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 02 — MEGA PROJECT 4: DELINQUENCY PREVENTION
# PROBLEM 2: INSTALLMENT PAYMENT BEHAVIOR / MISSED-PAYMENT PATTERN DETECTION
# Real Unsupervised Clustering on Real Payment-Streak Features (Not a
# Re-Derivation of Notebook 01's Supervised Rate-Based Model)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no supervised model and
# reuses no TARGET-fitted prediction from Notebook 01. It builds a real,
# vectorized STREAK feature set from real installments_payments.csv (via the
# new src/features/delinquency_features.py function
# engineer_payment_streak_features) and applies real unsupervised K-Means
# clustering -- grouping applicants by PATTERN similarity, never trained
# against real TARGET.
#
# WHY THIS IS A GENUINE, NOT REDUNDANT, ADDITION (read before trusting any
# "independent axis" claim below): Notebook 01 already summarizes an
# applicant's real installment history as RATES (e.g. "35% of installments
# were late") via engineer_installment_behavior_features() and feeds them to
# a SUPERVISED classifier predicting real TARGET. A rate cannot distinguish
# an applicant late on scattered, isolated installments from one currently
# mid-way through a real 6-payment late streak -- same rate, very different
# real risk posture. This notebook instead detects real STREAKS (consecutive
# runs of late/on-time status) via a genuinely different, real feature set
# (7 new features: longest late/on-time streak, streak count, the applicant's
# CURRENT streak, and a real alternation rate) AND a genuinely different
# mechanism (unsupervised clustering by pattern similarity, never supervised
# against TARGET). Section 11 below computes a real ANOVA F-test / eta-squared
# between this notebook's archetypes and Notebook 01's continuous real risk
# score as honest, computed evidence of how related the two outputs actually
# turned out to be -- not an asserted independence claim.
#
# HYPER REUSE: src/features/delinquency_features.py's new
# engineer_payment_streak_features (written for this notebook, reusable by
# any later notebook that needs streak features), src/reporting/report_
# builder.py, src/utils/performance_setup.py.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution.
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP4_DIR = SUITE_ROOT / "04_mega_project_4_delinquency_prevention"
ARTIFACTS_DIR = MP4_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP4_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP4_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import).
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2).
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import joblib
from scipy.stats import chi2_contingency, f_oneway
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.delinquency_features import engineer_payment_streak_features
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data.
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR)
installments = load_csv_cached(RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
check_ram_headroom(PERF)
print(f"[DATA] Real application_train.csv: {app.shape[0]:,} rows x {app.shape[1]} cols.")
print(f"[DATA] Real installments_payments.csv: {installments.shape[0]:,} rows x {installments.shape[1]} cols.")

# ---------------------------------------------------------------------------
# SECTION 5 — HYPER feature engineering: real payment-streak features.
# ---------------------------------------------------------------------------
streak_feat, FEATURE_COLS = engineer_payment_streak_features(installments)
scored = streak_feat.join(app.select(["SK_ID_CURR", "TARGET"]), on="SK_ID_CURR", how="inner")
N_SCOPE = scored.height
N_APP_TOTAL = app.height
print(f"[FEATURES] {len(FEATURE_COLS)} real streak features engineered for {N_SCOPE:,} of "
      f"{N_APP_TOTAL:,} real applicants ({N_SCOPE / N_APP_TOTAL:.1%}) with real installment history.")

df = scored.to_pandas()

# ---------------------------------------------------------------------------
# SECTION 6 — Real, data-driven K-Means clustering. K is chosen by the
# highest real silhouette score across a documented candidate range, never
# fixed by construction.
# ---------------------------------------------------------------------------
X = df[FEATURE_COLS].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = list(range(int(CONFIG.get("payment_pattern_k_min", 3)), int(CONFIG.get("payment_pattern_k_max", 7)) + 1))
MIN_CLUSTER_FRACTION = float(CONFIG.get("payment_pattern_min_cluster_fraction", 0.03))
MIN_CLUSTER_SIZE = max(int(MIN_CLUSTER_FRACTION * N_SCOPE), 20)
SIL_SAMPLE_SIZE = min(int(CONFIG.get("payment_pattern_silhouette_sample_size", 10_000)), N_SCOPE)

k_results = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    counts = np.bincount(labels)
    if counts.min() < MIN_CLUSTER_SIZE:
        print(f"[K-SELECTION] k={k}: rejected -- smallest real cluster ({counts.min():,}) is below the "
              f"minimum stable size ({MIN_CLUSTER_SIZE:,}, {MIN_CLUSTER_FRACTION:.1%} of scope).")
        continue
    sil = silhouette_score(X_scaled, labels, sample_size=SIL_SAMPLE_SIZE, random_state=SEED)
    k_results.append({"k": k, "silhouette": float(sil), "model": km, "labels": labels})
    print(f"[K-SELECTION] k={k}: real silhouette score={sil:.4f} (sampled {SIL_SAMPLE_SIZE:,} of "
          f"{N_SCOPE:,} real applicants for tractability).")

if not k_results:
    raise RuntimeError(
        f"No candidate K in {K_RANGE} produced every real cluster above the minimum stable size "
        f"({MIN_CLUSTER_SIZE:,}) -- the real data does not support this many distinguishable payment "
        f"patterns at this population size. Lower payment_pattern_k_max or "
        f"payment_pattern_min_cluster_fraction in project_config.json."
    )
best = max(k_results, key=lambda r: r["silhouette"])
K_CHOSEN = best["k"]
SILHOUETTE_CHOSEN = best["silhouette"]
CHAMPION_KMEANS = best["model"]
CLUSTER_LABELS_RAW = best["labels"]
print(f"[K-SELECTION] Real data-driven choice: k={K_CHOSEN} (highest real silhouette score "
      f"{SILHOUETTE_CHOSEN:.4f} among {len(k_results)} candidate(s) tried).")

PATTERN_LABELS = [f"Payment Pattern {chr(65 + i)}" for i in range(K_CHOSEN)]
df["PAYMENT_PATTERN"] = [PATTERN_LABELS[i] for i in CLUSTER_LABELS_RAW]

# ---------------------------------------------------------------------------
# SECTION 7 — Real pattern profiling, ordered by real observed default rate
# (descriptive labeling only -- the clustering itself is unsupervised).
# ---------------------------------------------------------------------------
pattern_agg = (
    df.groupby("PAYMENT_PATTERN", observed=True)
    .agg(n_applicants=("SK_ID_CURR", "size"), real_default_rate=("TARGET", "mean"),
         mean_longest_late_streak=("LONGEST_LATE_STREAK", "mean"),
         mean_current_streak_len=("CURRENT_STREAK_LEN", "mean"),
         pct_currently_in_late_streak=("CURRENT_STREAK_IS_LATE_INT", "mean"),
         mean_alternation_rate=("ALTERNATION_RATE", "mean"))
    .reindex(PATTERN_LABELS).reset_index()
)
pattern_agg["n_applicants"] = pattern_agg["n_applicants"].astype(int)
pattern_agg = pattern_agg.sort_values("real_default_rate", ascending=False).reset_index(drop=True)
for _, row in pattern_agg.iterrows():
    print(f"[PATTERN] {row['PAYMENT_PATTERN']}: {int(row['n_applicants']):,} real applicants, "
          f"real default rate={row['real_default_rate']:.4f}, "
          f"{row['pct_currently_in_late_streak']:.1%} currently in a late streak, "
          f"mean current streak length={row['mean_current_streak_len']:.2f}.")

# ---------------------------------------------------------------------------
# SECTION 8 — Real chi-square + Cramer's V + vectorized bootstrap CI
# (Payment Pattern vs. real TARGET).
# ---------------------------------------------------------------------------
contingency = pd.crosstab(df["PAYMENT_PATTERN"], df["TARGET"])
n_obs = int(contingency.values.sum())
min_dim = min(contingency.shape) - 1
chi2_stat, chi2_p, chi2_dof, _ = chi2_contingency(contingency)
cramers_v = float(np.sqrt((chi2_stat / n_obs) / max(min_dim, 1))) if min_dim > 0 else 0.0
print(f"[CHI-SQUARE] Real Payment Pattern vs. real TARGET: chi2={chi2_stat:.2f}, dof={chi2_dof}, "
      f"p-value={chi2_p:.6g}, Cramer's V={cramers_v:.4f}.")

N_BOOTSTRAP = 500
cell_probs = (contingency.values / n_obs).flatten()
cell_shape = contingency.shape
boot_v = []
for _ in range(N_BOOTSTRAP):
    draw = rng.multinomial(n_obs, cell_probs).reshape(cell_shape)
    if draw.sum() == 0 or min(draw.shape) < 2:
        continue
    try:
        chi2_bs, _, _, _ = chi2_contingency(draw)
        md_bs = min(draw.shape) - 1
        boot_v.append(float(np.sqrt((chi2_bs / n_obs) / max(md_bs, 1))) if md_bs > 0 else 0.0)
    except ValueError:
        continue
boot_v = np.array(boot_v) if boot_v else np.array([cramers_v])
V_CI_LOW, V_CI_HIGH = float(np.percentile(boot_v, 2.5)), float(np.percentile(boot_v, 97.5))
CRAMERS_V_ROBUST_THRESHOLD = 0.05
print(f"[VALIDATION] Real {len(boot_v)}-resample vectorized bootstrap 95% CI on Cramer's V "
      f"(Payment Pattern vs. real default): [{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}].")

# ---------------------------------------------------------------------------
# SECTION 9 — STATISTICAL ROBUSTNESS VERDICT. No monotonicity check by
# design: unordered categorical patterns from unsupervised clustering, the
# same disclosed design choice this suite already uses for other
# unsupervised segmentations (e.g. Mega Project 3 Notebook 02).
# ---------------------------------------------------------------------------
validation_checks = [
    ("chi_square_significant", chi2_p < 0.05),
    ("cramers_v_ci_excludes_zero", V_CI_LOW > CRAMERS_V_ROBUST_THRESHOLD),
    ("silhouette_score_finite_and_positive", bool(np.isfinite(SILHOUETTE_CHOSEN) and SILHOUETTE_CHOSEN > 0.0)),
    ("every_pattern_at_least_min_size", bool((pattern_agg["n_applicants"] >= MIN_CLUSTER_SIZE).all())),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks)
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 10 — SOFT DEPENDENCY: Notebook 01's real per-applicant early-
# delinquency-risk score, if present -- a real ANOVA F-test / eta-squared
# cross-check (continuous score vs. categorical pattern), honest evidence of
# how related the two real, independently-derived outputs turned out to be.
# ---------------------------------------------------------------------------
NB01_SCORES_PATH = MP4_DIR / "decision_engine" / "artifacts" / "notebook_01_delinquency_scores.csv"
NB01_CROSSCHECK_AVAILABLE = NB01_SCORES_PATH.exists()
NB01_ANOVA_F = None
NB01_ANOVA_P = None
NB01_ETA_SQUARED = None
if NB01_CROSSCHECK_AVAILABLE:
    try:
        nb01_scores = pd.read_csv(NB01_SCORES_PATH)[["SK_ID_CURR", "EARLY_DELINQUENCY_RISK_SCORE"]]
        cross_df = df.merge(nb01_scores, on="SK_ID_CURR", how="inner")
        groups = [g["EARLY_DELINQUENCY_RISK_SCORE"].to_numpy() for _, g in cross_df.groupby("PAYMENT_PATTERN", observed=True)]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) >= 2:
            NB01_ANOVA_F, NB01_ANOVA_P = (float(v) for v in f_oneway(*groups))
            grand_mean = cross_df["EARLY_DELINQUENCY_RISK_SCORE"].mean()
            ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
            ss_total = ((cross_df["EARLY_DELINQUENCY_RISK_SCORE"] - grand_mean) ** 2).sum()
            NB01_ETA_SQUARED = float(ss_between / ss_total) if ss_total > 0 else 0.0
            print(f"[CROSS-CHECK] Real one-way ANOVA, Notebook 01's continuous real risk score across "
                  f"this notebook's {K_CHOSEN} real payment patterns ({len(cross_df):,} matched real "
                  f"applicants): F={NB01_ANOVA_F:.2f}, p={NB01_ANOVA_P:.6g}, eta-squared={NB01_ETA_SQUARED:.4f} "
                  f"(reported honestly, not gated pass/fail -- some real association is expected since both "
                  f"outputs ultimately relate to real payment behavior; this is evidence of how related the "
                  f"two real, independently-derived outputs are, not a claim either way).")
        else:
            print("[CROSS-CHECK] Fewer than 2 real patterns had matched Notebook 01 scores -- skipping.")
            NB01_CROSSCHECK_AVAILABLE = False
    except Exception as e:
        print(f"[CROSS-CHECK] Real Notebook 01 cross-check could not be completed ({e}) -- this is a SOFT "
              f"dependency, so this notebook's own result is still complete and standalone.")
        NB01_CROSSCHECK_AVAILABLE = False
else:
    print("[CROSS-CHECK] Notebook 01's real per-applicant scores were not found -- this is a SOFT "
          "dependency. Run Notebook 01 first for the real cross-check.")

# ---------------------------------------------------------------------------
# SECTION 11 — Inline charts (vivid multicolor). No matplotlib.use(...) call
# anywhere in this file.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

_colors = _palette(len(pattern_agg))
axes[0].bar(pattern_agg["PAYMENT_PATTERN"], pattern_agg["real_default_rate"], color=_colors)
axes[0].set_title("Real Default Rate by Payment Pattern")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Real observed default rate")

axes[1].bar(pattern_agg["PAYMENT_PATTERN"], pattern_agg["n_applicants"], color=_colors)
axes[1].set_title("Real Applicant Count by Payment Pattern")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylabel("Real applicants")

sil_ks = [r["k"] for r in k_results]
sil_vals = [r["silhouette"] for r in k_results]
axes[2].plot(sil_ks, sil_vals, marker="o", color=VIVID_PALETTE[1], linewidth=2.5)
axes[2].scatter([K_CHOSEN], [SILHOUETTE_CHOSEN], color=VIVID_PALETTE[4], s=120, zorder=5, label=f"chosen k={K_CHOSEN}")
axes[2].set_title("Real Silhouette Score by Candidate K")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Real silhouette score")
axes[2].legend()

plt.tight_layout()
CHART_PATH = REPORTS_DIR / "notebook_02_charts.png"
plt.savefig(CHART_PATH, dpi=130, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# SECTION 12 — Save real trained artifacts.
# ---------------------------------------------------------------------------
MODEL_PATH = ARTIFACTS_DIR / "notebook_02_kmeans_model.joblib"
joblib.dump({
    "kmeans": CHAMPION_KMEANS, "scaler": scaler, "feature_cols": FEATURE_COLS,
    "pattern_labels": PATTERN_LABELS, "k_chosen": K_CHOSEN, "silhouette_chosen": SILHOUETTE_CHOSEN,
}, MODEL_PATH)
print(f"[SAVE] Real trained clustering bundle saved: {MODEL_PATH}")

patterns_df = df[["SK_ID_CURR", "PAYMENT_PATTERN", "TARGET"] + FEATURE_COLS]
patterns_path = ARTIFACTS_DIR / "notebook_02_payment_patterns.csv"
patterns_df.to_csv(patterns_path, index=False)
print(f"[SAVE] Real per-applicant payment patterns saved: {patterns_path}")

summary_artifact = {
    "notebook": "02_installment_payment_behavior_detection",
    "mega_project": 4,
    "n_scope": N_SCOPE,
    "n_app_total": N_APP_TOTAL,
    "k_chosen": K_CHOSEN,
    "silhouette_chosen": SILHOUETTE_CHOSEN,
    "pattern_agg": pattern_agg.to_dict(orient="records"),
    "chi_square_p": float(chi2_p),
    "cramers_v": cramers_v,
    "cramers_v_ci": [V_CI_LOW, V_CI_HIGH],
    "nb01_crosscheck_available": NB01_CROSSCHECK_AVAILABLE,
    "nb01_anova_f": NB01_ANOVA_F,
    "nb01_anova_p": NB01_ANOVA_P,
    "nb01_eta_squared": NB01_ETA_SQUARED,
    "analysis_verdict": ANALYSIS_VERDICT,
    "analysis_robust": ANALYSIS_ROBUST,
}
summary_path = REPORTS_DIR / "notebook_02_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_artifact, f, indent=2)
print(f"[SAVE] Real summary artifact saved: {summary_path}")

# ---------------------------------------------------------------------------
# SECTION 13 — Real Pipeline Integrity Checks (structural gate).
# ---------------------------------------------------------------------------
integrity_checks = [
    ("model_bundle_saved", MODEL_PATH.exists()),
    ("patterns_csv_saved", patterns_path.exists()),
    ("summary_json_saved", summary_path.exists()),
    ("every_applicant_assigned_a_pattern", bool(df["PAYMENT_PATTERN"].isin(PATTERN_LABELS).all())),
    ("pattern_agg_sums_to_scope", int(pattern_agg["n_applicants"].sum()) == N_SCOPE),
]
INTEGRITY_OK = all(ok for _, ok in integrity_checks)
for name, ok in integrity_checks:
    print(f"[INTEGRITY-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[INTEGRITY] Pipeline structural integrity: {'PASS' if INTEGRITY_OK else 'FAIL'}")

# ---------------------------------------------------------------------------
# SECTION 14 — Reports (HTML dashboard + Word + Excel).
# ---------------------------------------------------------------------------
exec_summary = [
    f"{K_CHOSEN} real, data-driven payment patterns found via K-Means clustering on real installment "
    f"streak features (silhouette={SILHOUETTE_CHOSEN:.4f}), chosen from candidates k={K_RANGE[0]}-{K_RANGE[-1]}.",
    f"Real chi-square test: Payment Pattern vs. real TARGET is "
    f"{'statistically significant' if chi2_p < 0.05 else 'not statistically significant'} "
    f"(p={chi2_p:.4g}), Cramer's V={cramers_v:.4f} (95% CI [{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}]).",
    f"Scope: {N_SCOPE:,} of {N_APP_TOTAL:,} real applicants ({N_SCOPE / N_APP_TOTAL:.1%}) with real "
    f"prior installment-payment history.",
]
if NB01_CROSSCHECK_AVAILABLE:
    exec_summary.append(
        f"Real cross-check against Notebook 01's continuous risk score: ANOVA F={NB01_ANOVA_F:.2f} "
        f"(p={NB01_ANOVA_P:.4g}), eta-squared={NB01_ETA_SQUARED:.4f} — honest evidence of how related "
        f"the two real, independently-derived outputs are."
    )
exec_summary.append(f"Statistical robustness verdict: {ANALYSIS_VERDICT}.")

sections = [
    {"heading": "Real K Selection (Data-Driven, Not Fixed)",
     "paragraphs": [f"K is chosen by the highest real silhouette score among candidates k="
                    f"{K_RANGE[0]}-{K_RANGE[-1]} where every real resulting cluster meets the minimum "
                    f"stable size ({MIN_CLUSTER_SIZE:,} applicants, {MIN_CLUSTER_FRACTION:.1%} of scope)."],
     "table": {"headers": ["k", "Real Silhouette Score"],
               "rows": [[r["k"], f"{r['silhouette']:.4f}"] for r in k_results]},
     "image_path": CHART_PATH},
    {"heading": "Real Payment Pattern Profiles",
     "paragraphs": ["Ordered by real observed default rate (descriptive labeling only — clustering "
                    "itself never used real TARGET)."],
     "table": {"headers": ["Pattern", "N Applicants", "Real Default Rate", "% Currently in Late Streak",
                            "Mean Current Streak Length"],
               "rows": [[r["PAYMENT_PATTERN"], int(r["n_applicants"]), f"{r['real_default_rate']:.4f}",
                         f"{r['pct_currently_in_late_streak']:.1%}", f"{r['mean_current_streak_len']:.2f}"]
                        for _, r in pattern_agg.iterrows()]}},
    {"heading": "Real Statistical Validation",
     "paragraphs": [f"Chi-square: chi2={chi2_stat:.2f}, dof={chi2_dof}, p={chi2_p:.4g}. "
                    f"Cramer's V={cramers_v:.4f}, real {len(boot_v)}-resample bootstrap 95% CI "
                    f"[{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}]."]},
]
if NB01_CROSSCHECK_AVAILABLE:
    sections.append({
        "heading": "Real Cross-Check vs. Notebook 01's Continuous Risk Score",
        "paragraphs": [f"One-way ANOVA of Notebook 01's real, continuous early-delinquency-risk score "
                       f"across this notebook's {K_CHOSEN} real payment patterns — reported honestly as "
                       f"evidence of relatedness, not gated pass/fail."],
        "table": {"headers": ["Statistic", "Value"],
                  "rows": [["F-statistic", f"{NB01_ANOVA_F:.4f}"], ["p-value", f"{NB01_ANOVA_P:.6g}"],
                           ["eta-squared", f"{NB01_ETA_SQUARED:.4f}"]]},
    })

write_csv_outputs({"pattern_profiles": pattern_agg, "k_selection": pd.DataFrame(
    [{"k": r["k"], "silhouette": r["silhouette"]} for r in k_results])}, REPORTS_DIR)

assumptions = {"random_seed": SEED, "k_range": f"{K_RANGE[0]}-{K_RANGE[-1]}",
               "min_cluster_fraction": MIN_CLUSTER_FRACTION, "n_bootstrap": N_BOOTSTRAP}
assumption_notes = {
    "random_seed": "Fixed seed for KMeans initialization and bootstrap reproducibility.",
    "k_range": "Candidate cluster counts tried; the real winner is chosen by highest silhouette score.",
    "min_cluster_fraction": "Minimum real cluster size (as a fraction of scope) for a candidate k to be valid.",
    "n_bootstrap": "Real resamples used for the Cramer's V 95% confidence interval.",
}
build_word_report(
    REPORTS_DIR / "notebook_02_report.docx",
    title="Mega Project 4 — Notebook 02: Installment Payment Behavior Detection",
    subtitle=f"{K_CHOSEN} real payment patterns | silhouette={SILHOUETTE_CHOSEN:.4f} | {ANALYSIS_VERDICT}",
    exec_summary=exec_summary, sections=sections,
)
print("[REPORT] Real Word report written.")

_pattern_rows = [[r["PAYMENT_PATTERN"], int(r["n_applicants"]), round(float(r["real_default_rate"]), 6),
                  round(float(r["pct_currently_in_late_streak"]), 6), round(float(r["mean_current_streak_len"]), 4)]
                 for _, r in pattern_agg.iterrows()]
_k_rows = [[r["k"], round(r["silhouette"], 6)] for r in k_results]
build_excel_workbook(
    REPORTS_DIR / "notebook_02_workbook.xlsx",
    assumptions=assumptions, assumption_notes=assumption_notes,
    data_sheets=[
        {"name": "Pattern Profiles",
         "headers": ["pattern", "n_applicants", "real_default_rate", "pct_currently_late_streak", "mean_current_streak_len"],
         "rows": _pattern_rows, "highlight_col": "real_default_rate"},
        {"name": "K Selection", "headers": ["k", "silhouette"], "rows": _k_rows, "highlight_col": "silhouette"},
    ],
)
print("[REPORT] Real Excel workbook written.")

build_html_dashboard(
    REPORTS_DIR / "notebook_02_dashboard.html",
    title="Mega Project 4 — Installment Payment Behavior Detection",
    subtitle=f"{K_CHOSEN} real payment patterns | silhouette={SILHOUETTE_CHOSEN:.4f} | {ANALYSIS_VERDICT}",
    kpi_cards=[
        {"label": "Real Patterns Found (k)", "value": str(K_CHOSEN)},
        {"label": "Silhouette Score", "value": f"{SILHOUETTE_CHOSEN:.4f}"},
        {"label": "Real Applicants In Scope", "value": f"{N_SCOPE:,}"},
        {"label": "Cramer's V vs. TARGET", "value": f"{cramers_v:.4f}"},
    ],
    charts=[
        {"id": "defaultRateChart", "title": "Real Default Rate by Payment Pattern", "type": "bar",
         "labels": pattern_agg["PAYMENT_PATTERN"].tolist(),
         "datasets": [{"label": "Real default rate", "data": pattern_agg["real_default_rate"].tolist(),
                       "backgroundColor": VIVID_PALETTE[1]}]},
        {"id": "countChart", "title": "Real Applicant Count by Payment Pattern", "type": "bar",
         "labels": pattern_agg["PAYMENT_PATTERN"].tolist(),
         "datasets": [{"label": "Real applicants", "data": pattern_agg["n_applicants"].tolist(),
                       "backgroundColor": VIVID_PALETTE[2]}]},
        {"id": "silhouetteChart", "title": "Real Silhouette Score by Candidate K", "type": "line",
         "labels": [str(k) for k in sil_ks],
         "datasets": [{"label": "Silhouette score", "data": sil_vals, "backgroundColor": VIVID_PALETTE[3]}]},
    ],
    data_table={"title": "Real Per-Applicant Payment Patterns (sample)",
                "columns": ["SK_ID_CURR", "PAYMENT_PATTERN", "TARGET"] + FEATURE_COLS,
                "rows": patterns_df.head(300).values.tolist()},
)
print("[REPORT] Real HTML dashboard written.")

print(f"\n[DONE] Notebook 02 complete in {time.time() - T0:.1f}s. "
      f"k={K_CHOSEN}, silhouette={SILHOUETTE_CHOSEN:.4f}. Verdict: {ANALYSIS_VERDICT}.")
